<a href="https://colab.research.google.com/github/chelseanbr/aih-final-project/blob/main/notebooks/02-generate-mcqs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning a Large Language Model for Layperson Medical Question Answering
AI in Healthcare | MSAI Spr '25 | Chelsea Ramos

## Objectives
1. Explore and sample the MedQuAD dataset
2. ***Generate medical multiple-choice questions (MCQs) using MedAlpaca***
3. Fine-tune TinyLLaMA with the generated MCQs
4. Evaluate model accuracy on held-out MCQs
5. Compare against other models (baseline and zero-shot)

---

# 2. Generate MCQs using Phi-2

In [13]:
import json
import re
from pathlib import Path
from tqdm import tqdm
from google.colab import files

# Upload JSONL file with question and answer fields
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

Saving medquad_sampled.jsonl to medquad_sampled (1).jsonl


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# checkpoint = "microsoft/phi-2"
# checkpoint = "stanford-crfm/BioMedLM"
checkpoint = "medalpaca/medalpaca-7b"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Load and parse JSONL file
with open(file_path, "r") as f:
    qa_data = [json.loads(line) for line in f]

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(checkpoint, torch_dtype=torch.float16).to(device)
model.eval()
print('Tokenizer and model loaded.')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/9.89G [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:609: UserWarning: `pad_token_id` should be positive but got -1. This will cause errors when batch generating, if there is padding. Please set `pad_token_id` explicitly as `model.generation_config.pad_token_id=PAD_TOKEN_ID` to avoid errors in generation
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Tokenizer and model loaded.


In [43]:
# Check if output file already exists
output_file = Path("generated_mcqs.jsonl")
if output_file.exists():
    response = input(f"File {output_file} exists. Delete it? (y/n): ")
    if response.lower().startswith("y"):
        output_file.unlink()
    else:
        raise RuntimeError("Aborting to avoid overwriting the file.")

# Few-shot example to encourage structure
few_shot_example = """
Question: What is asthma?
Answer: Asthma is a lung condition that causes airway inflammation and breathing difficulty.
<choices>
A. A skin condition
B. A viral infection
C. A lung disease that causes airway inflammation
D. A type of heart failure
</choices>
<reason>Asthma is a lung condition that causes airway inflammation and breathing difficulty.</reason>
<answer>C</answer>
"""

# Format QA into prompt
def format_prompt(question, answer):
    return f"""
Create multiple-choice answer choices and reasoning for the given medical Q&A pair.
Include 4 total choices: one correct and three plausible, but incorrect. Follow the XML-style structure below:
{few_shot_example}
Now it's your turn:
Question: {question}
Answer: {answer}
"""

tokenizer.padding_side = "right"

# Batch-generate MCQs
BATCH_SIZE = 2
NUM_RETURN_SEQUENCES = 3
num_prompts = 0
num_success = 0
# for i in tqdm(range(0, BATCH_SIZE*4+1, BATCH_SIZE),
#               desc=f"LLM Running on Micro Batches {BATCH_SIZE}"):  # DEBUG
for i in tqdm(range(0, len(qa_data), BATCH_SIZE),
                    desc=f"LLM Running on Micro Batches {BATCH_SIZE}"):
    batch = qa_data[i:i + BATCH_SIZE]
    prompts = [format_prompt(q["question"], q["answer"]) for q in batch]

    inputs = tokenizer(prompts, return_tensors="pt",
                       truncation=True,
                       max_length=250,
                       padding=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.7,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            num_return_sequences=NUM_RETURN_SEQUENCES,
        )

    prompt_lengths = [len(x) for x in inputs["input_ids"]]

    for j in range(len(batch)):
        num_prompts += 1
        for k in range(NUM_RETURN_SEQUENCES):
            idx = j * NUM_RETURN_SEQUENCES + k
            generated_tokens = outputs[idx][prompt_lengths[j]:]  # Strip prompt
            completion = tokenizer.decode(generated_tokens, skip_special_tokens=True)

            mcq_choices = re.search(r"<choices>(.*?)</choices>", completion, re.DOTALL)
            mcq_reason = re.search(r"<reason>(.*?)</reason>", completion, re.DOTALL)
            mcq_answer = re.search(r"<answer>(.*?)</answer>", completion, re.DOTALL)
            four_choices = 'A. ' in completion and 'B. ' in completion and 'C. ' in completion and 'D. ' in completion

            if not all([mcq_choices, mcq_reason, mcq_answer, four_choices]):
                # print(f"Skipping incomplete output [Batch {i}, Prompt {j}, Attempt {k+1}]:\n")
                continue
            else:
                # Add to output json
                entry = {
                      "question": batch[j]["question"],
                      "mcq_choices": mcq_choices.group(1).strip(),
                      "mcq_reason": mcq_reason.group(1).strip(),
                      "mcq_answer": mcq_answer.group(1).strip(),
                      "raw_completion": completion.strip()
                }
                with open(output_file, "a") as f:
                    f.write(json.dumps(entry) + "\n")
                # print(f"\nSuccess: Batch {i}, Prompt {j}, Attempt {k+1}:")
                num_success += 1
                break
    print(f"\n{num_success} / {num_prompts} ({num_success/num_prompts * 100:.2f}%) generated MCQs saved to {output_file}")

print(f"\nDone! {num_success} / {num_prompts} ({num_success/num_prompts * 100:.2f}%) generated MCQs saved to {output_file}")

File generated_mcqs.jsonl exists. Delete it? (y/n): y


TypeError: range() takes no keyword arguments